In [1]:
import json 
import time 
from typing import Optional 

import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM 


In [2]:


# this is required for the model to run on the GPU if available
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif device == "mps":
    print("Apple Silicon GPU (MPS)")
else:
    print("No GPU available")

"""
This function is used to synchronize the GPU.
It is required to ensure that the GPU is properly synchronized with the CPU.
Otherwise, the model will not be able to run on the GPU.
"""
def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elif torch.backends.mps.is_available():
        torch.mps.synchronize()



Using device: mps
Apple Silicon GPU (MPS)


In [3]:
start_time = time.time()
Model_id = "Qwen/Qwen3-4B"

# for loading the tokenizer specific to this model. We also do this in our embedding gen pipeline at Uber. This exists inside the model class
tokenizer = AutoTokenizer.from_pretrained(Model_id)


model = AutoModelForCausalLM.from_pretrained(Model_id, dtype=torch.float16, device_map="auto").eval()

print(f"Loaded model {Model_id} in {time.time() - start_time:.2f} seconds")




Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded model Qwen/Qwen3-4B in 4.78 seconds


In [4]:
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [24]:
def prepare_inputs(prompt: str):
    # TODO 1: create system and user messages, then render the generation prompt
    messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": prompt}]


    # chat_long = [
    #     {"role": "system", "content": "You are a helpful assistant."},
    #     {"role": "user", "content": "Can you give me a really long story about a space pirate?"},
    # ]

    """
    tokenizer.apply_chat_template() is a method in the Hugging Face Transformers library used to format conversational inputs into a single tokenizable string that matches the exact prompt structure a specific chat model was trained on. [1] (https://huggingface.co/docs/transformers/chat_templating), [2] (https://qwen.readthedocs.io/en/v1.5/inference/chat.html)
    """
    inputs = tokenizer.apply_chat_template([messages], add_generation_prompt=True, return_tensors="pt", return_dict = True, padding = True)

    # device_map="auto" can place the model on mps/cuda, so the input tensors
    # have to follow it or the embedding lookup fails
    inputs = inputs.to(model.device)


    print(inputs["attention_mask"].shape)

    inputs1 = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict = True)
    #inputs2 = tokenizer.apply_chat_template(chat_long, add_generation_prompt=True, return_tensors="pt", return_dict = True)

    inputs1 = inputs1.to(model.device)
    #inputs2 = inputs2.to(model.device)

    print(inputs1["attention_mask"].shape)
    #print(inputs2["attention_mask"].shape)
    
    # print(inputs)
    
    # print(inputs["input_ids"].__class__)

    # print(inputs["input_ids"].shape)

    # print(tokenizer.decode(inputs["input_ids"][0]))

    # print(inputs["input_ids"][0][1])

    # print(tokenizer.decode(inputs["input_ids"][0][26]))
    # Inputs will be a dictionary, with one key "input_ids" and a tensor as the value. The tensor will have the shape (1, 26), and another will be "attention_mask" with the same shape.
    # tensor is just all the unique token ids for the input prompt in the order they appear.
    # attention mask is just a binary mask of the same length as the input prompt, with 1s for all the tokens that are not padding tokens.
    assert messages, "Add the prompt messages"
    assert inputs is not None, "Render the prompt with the tokenizer"
    return inputs


x = prepare_inputs("aayein you maa ka bhosada")
print(x)



torch.Size([1, 29])
torch.Size([1, 29])
{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,     64,  59955,    258,    498,
            296,   5305,  16502,    293,  41835,   2584, 151645,    198, 151644,
          77091,    198]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]], device='mps:0')}


In [25]:

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
]

# Format chat template without tokenizing
formatted_chat = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_dict = True
)
print(formatted_chat)


{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 9707, 0, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [26]:
def select_next_token(
    logits: torch.Tensor,
    temperature: float,
    top_k: Optional[int] = None,
) -> torch.Tensor:
    if temperature == 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    # TODO 2: scale logits, apply the top-k cutoff, and sample one token.
    scaled_logits = logits / temperature 
    breakpoint()
    if top_k is not None:
        top_values = torch.topk(scaled_logits, top_k, dim=-1).values
        # keep the trailing dim so this broadcasts over the vocab, and use
        # masked_fill so the -inf stays on the model's device
        cutoff = top_values[:, -1:]
        scaled_logits = scaled_logits.masked_fill(scaled_logits < cutoff, float("-inf"))

    probabilities = scaled_logits.softmax(dim=-1)
    next_token = torch.multinomial(probabilities, num_samples=1)
    assert next_token is not None, "Sample one token"
    return next_token


We divide logits by temperature ($T$) before the Softmax Function to scale the distance between scores, which controls how peaky or flat the final probability distribution becomes during token sampling. [1, 2]  
Logits are raw, unnormalized prediction scores outputted by a language model for every possible next token. The division scales these numbers before exponentiation ($e^x$). [1, 3]  
How the Math Works with Exponentiation 
The Softmax formula with temperature divides each logit ($z_i$) by $T$ before exponentiating: $P_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$ 
Because exponentiation rapidly magnifies differences, dividing or multiplying the input value changes the relative gaps between options dramatically: [2, 3]  
Low Temperature ($T < 1$) 

• What happens: Dividing by a fraction (e.g., $0.5$) makes the raw logit values larger and magnifies the numerical gaps between them. 
• The effect: Exponentiating larger gaps causes the highest probability token to dwarf all others, creating a peaky, sharp distribution. 
• The result: The model becomes deterministic and nearly greedy, always picking the most obvious, predictable word. [1, 4, 5, 6, 7]  

High Temperature ($T > 1$) 

• What happens: Dividing by a number greater than $1$ (e.g., $2.0$) makes the logit values smaller and compresses the gaps between them. 
• The effect: Exponentiating smaller gaps results in a flat, uniform distribution where less likely tokens retain a much stronger competitive chance. 
• The result: The model becomes random and creative, exploring unusual word choices. [1, 3, 6, 7]  

If you want, I can explain:How top-k and top-p sampling work alongside temperatureWhy a temperature of 0 behaves like greedy decoding 
AI responses may include mistakes.

[1] https://www.youtube.com/watch?v=P0b9zdoFQYU
[2] https://ai-tldr.dev/learn/llm-fundamentals/text-generation/logits-explained/
[3] https://medium.com/@AdithyaGiridharan/temperature-top-k-in-llm-inference-what-actually-happens-inside-the-model-60d1769a9dce
[4] https://www.youtube.com/watch?v=IWSutLJbURU
[5] https://www.linkedin.com/posts/avi-chawla_temperature-in-llms-clearly-explained-activity-7391052572050092032-quW5
[6] https://kokosa.dev/blog/2026/temperature/
[7] https://medium.com/@shalinibs7076/how-temperature-really-works-inside-llms-3738d98e5c6b



In [40]:
@torch.inference_mode()
def generate_uncached(
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.0,
    top_k: Optional[int] = None,
) -> dict:
    inputs = prepare_inputs(prompt)
    generated_ids = inputs["input_ids"] # contains all the token ids for the input prompt in the order they appear
    # print(f"generated_ids: {generated_ids}") 
    attention_mask = inputs["attention_mask"] # contains a binary mask of the same length as the input prompt, with 1s for all the tokens that are not padding tokens.
    prompt_length = generated_ids.shape[1] # length of the input prompt after tokenization
    first_token_at = None

    sync_gpu() # synchronize the GPU - this is required to ensure that the GPU is properly synchronized with the CPU. Otherwise, the model will not be able to run on the GPU.
    started_at = time.perf_counter()
    for i in range(max_new_tokens):
        outputs = model(
            input_ids=generated_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
        """
        What is inside outputs? 
        The outputs object is an instance of CausalLMOutputWithPast. 
        Because you called the model directly, it contains:
        
        logits: A massive tensor of shape (batch_size, sequence_length, vocab_size). 
        This contains the unnormalized prediction scores for every single token position in your input. 
        If you want to know what the model thinks the next token should be, 
        you look at the very last position of these logits.
        
        loss: This will be None because you didn't pass a labels argument to the model.
        
        past_key_values: This will be None because you set use_cache=False.
        """
        # print(f"outputs: {outputs}")

        # print(f"outputs.logits: {outputs.logits.shape}")

        # TODO 3: select and append the next token, extend the mask, and stop on EOS.
        breakpoint()
        next_token = select_next_token(
            outputs.logits[:, -1, :], # passing in the last token's logits. The last token's logits contain raw scores for the entire vocabulary
            temperature,
            top_k,
        )
        
        if first_token_at is None: # this is used to calculate the time it takes to generate the first token
            sync_gpu()
            first_token_at = time.perf_counter()


        print(tokenizer.decode(next_token))
        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
        print(next_token.__class__)
        print(f"i: {i}, next_token: {  next_token}")

        if next_token.item() == tokenizer.eos_token_id:
            break

    sync_gpu()
    finished_at = time.perf_counter()
    completion_ids = generated_ids[0, prompt_length:]
    return {
        "text": tokenizer.decode(completion_ids, skip_special_tokens=True),
        "token_ids": completion_ids.tolist(),
        "input_tokens": prompt_length,
        "ttft_ms": (first_token_at - started_at) * 1000,
        "latency_ms": (finished_at - started_at) * 1000,
    }

In [41]:
breakpoint()
uncached_output = generate_uncached("do robots dream of electric sheep?")

print(json.dumps(uncached_output, indent=2))

torch.Size([1, 26])
torch.Size([1, 26])
['<think>']
<class 'torch.Tensor'>
i: 0, next_token: tensor([[151667]], device='mps:0')
['\n']
<class 'torch.Tensor'>
i: 1, next_token: tensor([[198]], device='mps:0')
['Okay']
<class 'torch.Tensor'>
i: 2, next_token: tensor([[32313]], device='mps:0')
[',']
<class 'torch.Tensor'>
i: 3, next_token: tensor([[11]], device='mps:0')
[' the']
<class 'torch.Tensor'>
i: 4, next_token: tensor([[279]], device='mps:0')
[' user']
<class 'torch.Tensor'>
i: 5, next_token: tensor([[1196]], device='mps:0')
[' asked']
<class 'torch.Tensor'>
i: 6, next_token: tensor([[4588]], device='mps:0')
[',']
<class 'torch.Tensor'>
i: 7, next_token: tensor([[11]], device='mps:0')
[' "']
<class 'torch.Tensor'>
i: 8, next_token: tensor([[330]], device='mps:0')
['Do']
<class 'torch.Tensor'>
i: 9, next_token: tensor([[5404]], device='mps:0')
[' robots']
<class 'torch.Tensor'>
i: 10, next_token: tensor([[28707]], device='mps:0')
[' dream']
<class 'torch.Tensor'>
i: 11, next_token:

In [42]:
%debug

> /var/folders/zw/sqk2snt97jqfq29p3w9tvk140000gq/T/ipykernel_10174/4213885269.py(60)generate_uncached()
     58         print(f"i: {i}, next_token: {  next_token}")
     59 
---> 60         if next_token.item() == tokenizer.eos_token_id:
     61             break
     62 



In [45]:
print(tokenizer.decode(151644))

<|im_start|>


In [46]:
@torch.inference_mode()
def generate_cached(
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.0,
    top_k: Optional[int] = None,
) -> dict:
    inputs = prepare_inputs(prompt)
    generated_ids = inputs["input_ids"] # contains all the token ids for the input prompt in the order they appear
    print(f"generated_ids: {generated_ids}") 
    attention_mask = inputs["attention_mask"] # contains a binary mask of the same length as the input prompt, with 1s for all the tokens that are not padding tokens.
    prompt_length = generated_ids.shape[1] # length of the input prompt after tokenization
    first_token_at = None

    sync_gpu() # synchronize the GPU - this is required to ensure that the GPU is properly synchronized with the CPU. Otherwise, the model will not be able to run on the GPU.
    started_at = time.perf_counter()
    outputs = model(
        input_ids=generated_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )


    past_key_values = outputs.past_key_values 

    for _ in range(max_new_tokens):
        """
        What is inside outputs? 
        The outputs object is an instance of CausalLMOutputWithPast. 
        Because you called the model directly, it contains:
        
        logits: A massive tensor of shape (batch_size, sequence_length, vocab_size). 
        This contains the unnormalized prediction scores for every single token position in your input. 
        If you want to know what the model thinks the next token should be, 
        you look at the very last position of these logits.
        
        loss: This will be None because you didn't pass a labels argument to the model.
        
        past_key_values: This will be None because you set use_cache=False.
        """

        # TODO 4: select the next token and reuse past_key_values for the next model call.
        next_token = select_next_token(
            outputs.logits[:, -1, :], # passing in the last token's logits. The last token's logits contain raw scores for the entire vocabulary
            temperature,
            top_k,
        )

        if next_token.item() == tokenizer.eos_token_id:
            break
        
        if first_token_at is None: # this is used to calculate the time it takes to generate the first token
            sync_gpu()
            first_token_at = time.perf_counter()


        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            use_cache=True,
            past_key_values=past_key_values,
        )

        past_key_values = outputs.past_key_values 
        

    sync_gpu()
    finished_at = time.perf_counter()
    completion_ids = generated_ids[0, prompt_length:]
    return {
        "text": tokenizer.decode(completion_ids, skip_special_tokens=True),
        "token_ids": completion_ids.tolist(),
        "input_tokens": prompt_length,
        "ttft_ms": (first_token_at - started_at) * 1000,
        "latency_ms": (finished_at - started_at) * 1000,
    }

In [47]:
cached_output = generate_cached("Do robots dream of electric sheep?")

print(json.dumps(cached_output, indent=2))

torch.Size([1, 26])
torch.Size([1, 26])
generated_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   5404,  28707,   7904,    315,
           9072,  31912,     30, 151645,    198, 151644,  77091,    198]],
       device='mps:0')
{
  "text": "<think>\nOkay, the user asked, \"Do robots dream of electric sheep?\" That's a classic question from the movie \"Blade Runner,\" right? I need to explain the origin of the question and its meaning.\n\nFirst, I should mention that the question is from the film \"Blade Runner,\" where replicants are bioengineered humans. The question is posed by the protagonist, Deckard, to test if the replicants are truly alive. The answer is that they don't dream of electric sheep because they're not humans, but the question is a metaphor for the search for meaning and humanity.\n\nI should explain the context of the movie, the concept of replicants, and the significance of the qu

In [ ]:
@torch.inference_mode()
def generate_stream(
    prompt: str,
    max_new_tokens: int = 32,
    temperature: float = 0.0,
    top_k: Optional[int] = None,
):
    """Yield SSE chunks as tokens are produced. FastAPI's StreamingResponse
    iterates this generator and flushes each yield to the client immediately.
    """
    if top_k is not None and top_k <= 0:
        top_k = None

    inputs = prepare_inputs(prompt)
    generated_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    prompt_length = generated_ids.shape[1]
    previous_text = ""

    outputs = model(
        input_ids=generated_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values

    for _ in range(max_new_tokens):
        next_token = select_next_token(
            outputs.logits[:, -1, :],
            temperature,
            top_k,
        )
        if next_token.item() == tokenizer.eos_token_id:
            break

        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones_like(next_token)],
            dim=1,
        )

        # Decode the whole completion so far, then emit only the new suffix.
        # Needed because one token is not always a complete UTF-8 / BPE piece.
        current_text = tokenizer.decode(
            generated_ids[0, prompt_length:],
            skip_special_tokens=True,
        )
        if not current_text.endswith("\ufffd"):
            chunk = current_text[len(previous_text):]
            previous_text = current_text
            if chunk:
                yield f"data: {json.dumps({'text': chunk})}\n\n"

        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            use_cache=True,
            past_key_values=past_key_values,
        )
        past_key_values = outputs.past_key_values

    yield "data: [DONE]\n\n"


In [ ]:
from pydantic import BaseModel, Field

class GenerateRequest(BaseModel):
    # TODO 7: add validated prompt, token limit, temperature, and top-k fields.
    prompt: str = Field(..., description="The prompt to generate from")
    token_limit: int = Field(..., description="The maximum number of tokens to generate")
    temperature: float = Field(..., description="The temperature to use for generation")
    top_k: int = Field(..., description="The top-k value to use for generation")

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.prompt = kwargs.get("prompt", "")
        self.token_limit = kwargs.get("token_limit", 32)
        self.temperature = kwargs.get("temperature", 0.0)
        self.top_k = kwargs.get("top_k", None)

class GenerateResponse(BaseModel):
    # TODO 7: expose text, timing, token count, throughput, and cache usage.
    text: str = Field(..., description="The generated text")
    timing: dict = Field(..., description="The timing information")
    token_count: int = Field(..., description="The number of tokens in the generated text")
    throughput: float = Field(..., description="The throughput of the generation")
    cache_usage: str = Field(..., description="The cache usage information")
    status_code: int = Field(..., description="The status code of the response")

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.text = kwargs.get("text", "")
        self.timing = kwargs.get("timing", {})
        self.token_count = kwargs.get("token_count", 0)
        self.throughput = kwargs.get("throughput", 0.0)
        self.cache_usage = kwargs.get("cache_usage", "")
    


In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI(title="tinyinference")

@app.get("/health")
def health():
    return {"status": "ok"}

# TODO 8: implement the measured generation and streaming endpoints.
@app.post("/generate", response_model=GenerateResponse)
def generate_text(request: GenerateRequest):
    response = generate_cached(
        request.prompt,
        request.token_limit,
        request.temperature,
        request.top_k,
    )

    return GenerateResponse(
        text=response["text"],
        timing={"latency_ms": response["latency_ms"], "ttft_ms": response["ttft_ms"]},
        token_count=response["input_tokens"],
        throughput=response.get("throughput", 0.0),
        cache_usage="true", # because we're calling generate_cached here 
        status_code=200,
    )

@app.post("/stream")
async def stream_text(request: GenerateRequest):
    # generate_stream() returns a generator. Do not list() or join() it here —
    # StreamingResponse iterates it and sends each yield as an SSE frame.
    return StreamingResponse(
        generate_stream(
            request.prompt,
            request.token_limit,
            request.temperature,
            request.top_k,
        ),
        media_type="text/event-stream",
    )


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# TODO 9: call the health and generation endpoints and verify their responses.
health_response = client.get("/health")
generation_response = client.post("/generate", json={"prompt": "Hello, world!", "token_limit": 32, "temperature": 0.0, "top_k": 0})

assert health_response.status_code == 200
assert health_response.json() == {"status": "ok"}

print(json.dumps(generation_response.json(), indent=2))
assert generation_response.status_code == 200

